# Clinical and Demographic Measurements for SMT and Tenecteplase Groups in Minor Anterior-Circulation Medium-Vessel Occlusion Stroke: Exploration with `mlcroissant`
This notebook provides a guided workflow for exploring the FAIR^2 clinical stroke dataset via the `mlcroissant` library. All metadata entities are referenced using their `@id`s for precision and reproducibility.

### Dataset Source
- The dataset is defined by a Croissant schema accessible at:

`https://sen.science/doi/10.71728/senscience.k6wm-4vww/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load Croissant metadata and preview dataset info. All referencing uses `@id` fields as shown in the schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.k6wm-4vww/fair2.json'

# Load metadata
dataset = mlc.Dataset(croissant_url)

# Access top-level metadata entity (as object, not dict)
meta = dataset.metadata
print(f"Dataset name: {meta.name}\nDescription: {meta.description}")
print(f"Version: {getattr(meta, 'version', 'N/A')}")
print(f"Published: {getattr(meta, 'datePublished', 'N/A')}")
if hasattr(meta, 'keywords'):
    print(f"Keywords: {', '.join(meta.keywords)}")

## 2. Data Overview
List all available record sets and their field (column) definitions using their `@id` fields. This enables reproducible referencing throughout analysis.

In [ ]:
# List all available record sets in the dataset
record_sets = list(dataset.record_sets())

print(f"Number of record sets: {len(record_sets)}\n")
record_set_ids = []

for rs in record_sets:
    print(f"Record Set: {rs.name} (@id={rs.id})")
    record_set_ids.append(rs.id)
    # Each record set may have fields (columns). List them with their @id
    if hasattr(rs, 'fields'):
        print("  Fields:")
        for field in rs.fields:
            # field may have 'name' and 'id'
            print(f"    - {field.name} (@id={field.id}, type={getattr(field, 'data_type', 'N/A')})")
    print()

if not record_set_ids:
    print("No record sets were found in this schema; please check the schema definition.")

## 3. Data Extraction
Extract rows for each record set by `@id`. Each is loaded into a pandas DataFrame. Use fields as shown above in the overview. (Access columns by their `@id`s!)

In [ ]:
dataframes = {}

for record_set_id in record_set_ids:
    # Load all records for this record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"DataFrame for RecordSet {record_set_id} - Shape: {df.shape}")
        print(f"Columns [by @id]: {df.columns.tolist()}\n")
    else:
        print(f"RecordSet {record_set_id} contains no records.")

# Let's demonstrate with the first populated record set
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Example records for RecordSet @id={main_record_set_id}")
    display(dataframes[main_record_set_id].head())
else:
    print("No tabular data extracted. Check schema/source.")

## 4. Exploratory Data Analysis (EDA)
Filter, normalize, and group data using field and record set `@id`s. Replace variable values below with those identified above!

Example: Filtering and normalizing a numeric column, and grouping by a categorical field.

In [ ]:
# Example: Set the RecordSet and field @ids for demonstration.
example_record_set_id = main_record_set_id  # Use main_record_set_id from above

df = dataframes[example_record_set_id]

"""
You must set these to match your data; list columns above if unsure.
"""
# Pick field IDs (as strings):
# Replace with actual @id from printed column list, e.g. 'age', 'sex', etc.
numeric_field_id = df.columns[df.dtypes.apply(lambda x: pd.api.types.is_numeric_dtype(x))][0] if not df.empty else None
group_field_id = df.columns[1] if len(df.columns) > 1 else None

# -- Replace above as needed for your column names/ids --

# Filtering: keep rows with value in numeric field > threshold
if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean()  # or pick a threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize (Z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping
    if group_field_id and group_field_id in df.columns:
        grouped_df = (
            filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame("mean_value")
        )
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize a numeric field (by its `@id`) and its grouping, if available. Update field IDs as needed from your dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histograms and boxplots for the numeric field
if numeric_field_id is not None and not df.empty:
    plt.figure(figsize=(9, 4))
    plt.subplot(1, 2, 1)
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True, color='dodgerblue')
    plt.title(f"Distribution of {numeric_field_id}")

    plt.subplot(1, 2, 2)
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df, palette='Set2')
    plt.title(f"{numeric_field_id} by {group_field_id}")

    plt.tight_layout()
    plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion
- The FAIR^2 dataset (referenced by all objects using their `@id`s) was loaded and record sets/fields explored.
- Extraction, filtering, and basic visualizations were demonstrated with proper use of field `@id`s so analyses are reproducible.
- Replace example field/group IDs in code blocks with your own, based on Data Overview output, to analyze specific variables of interest in the dataset.

**Note:** For all advanced data processing, always refer to columns/fields by their Croissant `@id`, not their display name, to maintain schema traceability!